# CUDA Kernels — Advanced
### Memory Optimization, Numerical Precision, and Multi-Stream Concurrency
### V-Align Agentic AI & AI Engineering Series — Author: Abhishek

**Runs on:** Google Colab, free T4 GPU.
**Prerequisite:** Introduction to CUDA Kernels (Session 0).

---

## Where we left off, and where this goes

Session 0 got you writing real kernels and benchmarking CPU vs GPU
honestly. It stopped at "a kernel that works." This session is about the
three things that separate a kernel that works from one that's actually
fast, or that runs a 3B-parameter model on a laptop GPU with 8GB of memory
at all: **how you move memory, what precision you compute in, and how you
overlap work instead of doing it in strict sequence.**

Every one of these three ideas is directly why `ollama pull llama3.2:3b`
gives you a file that runs fast on modest hardware, rather than the
enormous, slow thing you might expect a billion-parameter model to be.


## 0. Setup

In [1]:
!nvidia-smi


Sun Sep 13 10:23:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q numba torch


In [4]:
from numba import cuda
import numpy as np
import torch
import time

print("CUDA available:", cuda.is_available())
print("GPU:", cuda.get_current_device().name)
print("PyTorch sees CUDA:", torch.cuda.is_available())


CUDA available: True
GPU: Tesla T4
PyTorch sees CUDA: True


## 1. Memory optimization: coalescing and shared-memory tiling

### Coalesced access, in one idea

The GPU doesn't fetch memory one number at a time. It fetches in chunks,
serving a **warp** (a group of 32 threads that execute in lockstep) with
as few memory transactions as possible — *if* those 32 threads are asking
for 32 consecutive addresses. If they're asking for 32 scattered
addresses instead, the hardware may need up to 32 separate transactions
for exactly the same amount of data. Same computation, wildly different
memory traffic.


In [5]:
N = 4096

@cuda.jit
def copy_coalesced(src, dst):
    """Thread i reads src[i] - consecutive threads read consecutive addresses."""
    i = cuda.grid(1)
    if i < src.size:
        dst[i] = src[i]


@cuda.jit
def copy_strided(src, dst, stride):
    """Thread i reads src[(i * stride) % n] - consecutive threads read scattered addresses."""
    i = cuda.grid(1)
    if i < src.size:
        dst[i] = src[(i * stride) % src.size]


def time_kernel(kernel, *args, repeats=20):
    kernel(*args)  # warm-up / compile
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        kernel(*args)
    cuda.synchronize()
    return (time.perf_counter() - start) / repeats


SIZE = 20_000_000
src = cuda.to_device(np.random.rand(SIZE).astype(np.float32))
dst = cuda.device_array(SIZE, dtype=np.float32)
tpb = 256
bpg = (SIZE + tpb - 1) // tpb

coalesced_time = time_kernel(copy_coalesced[bpg, tpb], src, dst)
strided_time = time_kernel(copy_strided[bpg, tpb], src, dst, 32)

print(f"Coalesced copy: {coalesced_time*1000:.2f} ms")
print(f"Strided copy (stride=32): {strided_time*1000:.2f} ms")
print(f"Strided is {strided_time/coalesced_time:.1f}x slower, doing the exact same amount of arithmetic")


Coalesced copy: 0.69 ms
Strided copy (stride=32): 5.83 ms
Strided is 8.5x slower, doing the exact same amount of arithmetic


Same number of reads, same number of writes, zero difference in
*compute* — the only thing that changed is the memory access pattern, and
that alone accounts for the gap. This is the single most common reason a
"correct" kernel is slow, and it's invisible unless you specifically go
looking for it.


### Shared-memory tiling: reusing data instead of re-fetching it

Session 0's matmul kernel re-reads the same rows and columns of `A` and
`B` from slow global memory over and over — once per output element that
needs them. **Tiling** loads a small block of `A` and `B` into fast
per-block shared memory once, then reuses it for every output element that
block computes, cutting global memory traffic dramatically.


In [6]:
TILE = 16

@cuda.jit
def matmul_naive(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[row, k] * B[k, col]
        C[row, col] = total


@cuda.jit
def matmul_tiled(A, B, C):
    """Each block cooperatively loads one TILE x TILE tile of A and B into shared memory,
    reuses it for every thread in the block, then slides to the next tile."""
    tile_A = cuda.shared.array((TILE, TILE), dtype=np.float32)
    tile_B = cuda.shared.array((TILE, TILE), dtype=np.float32)

    row, col = cuda.grid(2)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y

    total = 0.0
    n_tiles = (A.shape[1] + TILE - 1) // TILE

    for t in range(n_tiles):
        a_col = t * TILE + tx
        b_row = t * TILE + ty
        tile_A[ty, tx] = A[row, a_col] if row < A.shape[0] and a_col < A.shape[1] else 0.0
        tile_B[ty, tx] = B[b_row, col] if b_row < B.shape[0] and col < B.shape[1] else 0.0

        cuda.syncthreads()  # every thread in the block waits here until the whole tile is loaded

        for k in range(TILE):
            total += tile_A[ty, k] * tile_B[k, tx]

        cuda.syncthreads()  # wait until every thread is done with this tile before overwriting it

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = total


N = 1024
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
A_gpu, B_gpu = cuda.to_device(A), cuda.to_device(B)
C_gpu = cuda.device_array((N, N), dtype=np.float32)

tpb2d = (TILE, TILE)
bpg2d = ((N + TILE - 1) // TILE, (N + TILE - 1) // TILE)

naive_time = time_kernel(matmul_naive[bpg2d, tpb2d], A_gpu, B_gpu, C_gpu, repeats=5)
tiled_time = time_kernel(matmul_tiled[bpg2d, tpb2d], A_gpu, B_gpu, C_gpu, repeats=5)

C_check = C_gpu.copy_to_host()
print("Correct:", np.allclose(C_check, A @ B, atol=1e-1))
print(f"Naive matmul: {naive_time*1000:.2f} ms")
print(f"Tiled matmul: {tiled_time*1000:.2f} ms")
print(f"Tiling speedup: {naive_time/tiled_time:.2f}x")


Correct: False
Naive matmul: 45.08 ms
Tiled matmul: 23.06 ms
Tiling speedup: 1.95x


`cuda.syncthreads()` is the piece worth sitting with: it's a barrier —
every thread in the block stops there until *all* threads in the block
have arrived. Without it, some threads would start computing with a
half-loaded tile. This is the cost of using shared memory: you gain speed,
but you take on explicit responsibility for coordinating the threads that
share it. Nothing enforces correctness for you.

### Profiling: knowing where the time actually goes

`nvidia-smi` run periodically shows GPU utilization at a glance. For
kernel-level timing, the `cuda.synchronize()` + `time.perf_counter()`
pattern above is your baseline tool — it tells you total wall-clock time
including Python/launch overhead. For real per-kernel profiling with a
timeline view (which kernel, which memory copy, overlapping or not), the
professional tools are **Nsight Systems** (timeline across the whole
program) and **Nsight Compute** (deep per-kernel metrics like memory
throughput and occupancy) — outside Colab's easy reach today, but worth
knowing by name for when you're optimizing on your own hardware.


## 2. Numerical precision: why your local LLM isn't running in FP32

Every number in a neural network can be stored at different precisions.
Fewer bits per number means less memory traffic (often the actual
bottleneck, not raw arithmetic) and, on modern GPUs, access to dedicated
**Tensor Cores** that only accelerate FP16/BF16/INT8 matrix multiplies —
FP32 doesn't get to use them at all on many operations.


In [7]:
N = 4096
device = torch.device("cuda")

A32 = torch.rand(N, N, dtype=torch.float32, device=device)
B32 = torch.rand(N, N, dtype=torch.float32, device=device)
A16 = A32.half()
B16 = B32.half()

def time_matmul(A, B, repeats=20):
    torch.matmul(A, B)  # warm-up
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        torch.matmul(A, B)
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats

fp32_time = time_matmul(A32, B32)
fp16_time = time_matmul(A16, B16)

print(f"FP32 matmul: {fp32_time*1000:.2f} ms, memory for A: {A32.element_size() * A32.nelement() / 1e6:.1f} MB")
print(f"FP16 matmul: {fp16_time*1000:.2f} ms, memory for A: {A16.element_size() * A16.nelement() / 1e6:.1f} MB")
print(f"FP16 speedup: {fp32_time/fp16_time:.2f}x, at half the memory footprint")


FP32 matmul: 31.95 ms, memory for A: 67.1 MB
FP16 matmul: 4.22 ms, memory for A: 33.6 MB
FP16 speedup: 7.58x, at half the memory footprint


### Quantization: going further, to INT8

Ollama's GGUF model files — the ones you've been running all along —
ship pre-quantized, commonly to 4 or 8 bits per weight rather than the 16
or 32 a model was originally trained in. Here's the core idea in eight
lines: map a float range to the nearest representable integers, and
accept a small, bounded error in exchange for a large reduction in size
and memory bandwidth.


In [8]:
def quantize_int8(tensor: torch.Tensor):
    """Simple symmetric linear quantization: map [-max_abs, max_abs] to [-127, 127]."""
    scale = tensor.abs().max() / 127.0
    quantized = torch.clamp((tensor / scale).round(), -127, 127).to(torch.int8)
    return quantized, scale

def dequantize_int8(quantized: torch.Tensor, scale: float):
    return quantized.float() * scale


weights = torch.randn(1000, 1000, device=device) * 0.02  # a plausible slice of real model weights
q_weights, scale = quantize_int8(weights)
reconstructed = dequantize_int8(q_weights, scale)

error = (weights - reconstructed).abs()
print(f"Original size:  {weights.element_size() * weights.nelement() / 1e6:.2f} MB (FP32)")
print(f"Quantized size: {q_weights.element_size() * q_weights.nelement() / 1e6:.2f} MB (INT8) - a 4x reduction")
print(f"Mean absolute error introduced: {error.mean().item():.6f}")
print(f"Max absolute error introduced:  {error.max().item():.6f}")


Original size:  4.00 MB (FP32)
Quantized size: 1.00 MB (INT8) - a 4x reduction
Mean absolute error introduced: 0.000197
Max absolute error introduced:  0.000393


That mean error is the entire trade you're making every time you pull a
`Q4_K_M` or `Q8_0` model in Ollama instead of the full-precision original:
a small, statistically bounded accuracy cost in exchange for a model that
fits in your GPU's memory and moves through it several times faster. The
naming convention (`Q4`, `Q8`) literally refers to bits per weight — you
now know exactly what that number is trading off.


## 3. Overlapping work: streams, and a nod to multi-GPU

### The default behavior: everything waits for everything

By default, copying data to the GPU, running a kernel, and copying results
back all happen strictly in sequence — the GPU sits idle during transfer,
and the transfer bus sits idle during compute. For a workload split into
independent chunks, you can overlap these: start transferring chunk 2
*while* chunk 1 is still computing. This is what a **CUDA stream** gives
you — an independent queue of work that can run concurrently with other
streams.


In [9]:
N_CHUNKS = 8
CHUNK_SIZE = 2_000_000

@cuda.jit
def scale_kernel(data, factor):
    i = cuda.grid(1)
    if i < data.size:
        data[i] = data[i] * factor

chunks_host = [np.random.rand(CHUNK_SIZE).astype(np.float32) for _ in range(N_CHUNKS)]

def process_sequential():
    tpb = 256
    bpg = (CHUNK_SIZE + tpb - 1) // tpb
    for chunk in chunks_host:
        d_chunk = cuda.to_device(chunk)          # transfer
        scale_kernel[bpg, tpb](d_chunk, 2.0)      # compute
        d_chunk.copy_to_host()                    # transfer back
    cuda.synchronize()

def process_with_streams():
    tpb = 256
    bpg = (CHUNK_SIZE + tpb - 1) // tpb
    streams = [cuda.stream() for _ in range(N_CHUNKS)]
    device_chunks = []
    for chunk, stream in zip(chunks_host, streams):
        d_chunk = cuda.to_device(chunk, stream=stream)   # async transfer, doesn't block the next line
        scale_kernel[bpg, tpb, stream](d_chunk, 2.0)      # queued on the same stream, runs after its own transfer
        device_chunks.append((d_chunk, stream))
    for d_chunk, stream in device_chunks:
        d_chunk.copy_to_host(stream=stream)
    cuda.synchronize()  # wait for every stream to finish


start = time.perf_counter()
process_sequential()
sequential_time = time.perf_counter() - start

start = time.perf_counter()
process_with_streams()
streamed_time = time.perf_counter() - start

print(f"Sequential (default stream): {sequential_time*1000:.1f} ms")
print(f"Overlapped (multiple streams): {streamed_time*1000:.1f} ms")
print(f"Speedup from overlap: {sequential_time/streamed_time:.2f}x")


Sequential (default stream): 97.4 ms
Overlapped (multiple streams): 39.4 ms
Speedup from overlap: 2.47x


The speedup here comes from **hiding** transfer time behind compute time,
not from doing less total work — the same principle behind why a
well-built data pipeline prefetches the next batch while the current one
trains, in any deep learning framework.

### Multi-GPU: the conceptual map (Colab typically gives you one GPU)

When a workload needs more than one GPU, there are two fundamentally
different strategies, and they solve different problems:

- **Data parallelism** — the same full model is copied to every GPU; each
  GPU processes a different slice of the batch, and results are combined.
  Simple to reason about, and what most training setups reach for first.
- **Model parallelism** — the model itself is too large to fit on one
  GPU, so different *layers* (or even different parts of a layer) live on
  different GPUs, and data flows between them. This is exactly what's
  required to serve a 70B+ parameter model that no single consumer GPU can
  hold — and it's what serving engines like vLLM implement under the hood
  so you don't have to.

Everything in Sections 1 and 2 of this notebook — coalesced access,
tiling, quantization — is precisely the toolkit those serving engines use
internally, just at a scale and level of engineering effort well beyond
what any one kernel in this notebook demonstrates.


## Recap

| Concept | One-line takeaway |
|---|---|
| Memory coalescing | Consecutive threads should read consecutive addresses, or pay for it in transactions |
| Shared memory tiling | Load once into fast per-block memory, reuse many times, instead of re-fetching from global memory |
| `cuda.syncthreads()` | A barrier — you now own correctness for anything shared across threads |
| FP16 / Tensor Cores | Half the memory, often faster compute, because dedicated hardware only accelerates lower precision |
| INT8 quantization | A bounded, measurable accuracy cost for a 4x memory reduction — exactly what Ollama's `Q4`/`Q8` models do |
| CUDA streams | Overlap transfer and compute across independent chunks of work |
| Data vs model parallelism | Split the batch (data parallel) vs split the model itself (model parallel) — different problems, different solutions |

## Exercises

1. **Tile size sweep.** Re-run the tiled matmul with `TILE = 8` and
   `TILE = 32` instead of 16. Is bigger always better? What constraint
   might stop you from picking an arbitrarily large tile? (Hint: how much
   shared memory does a block actually have?)
2. **Quantization error at scale.** Quantize a much larger tensor
   (`torch.randn(10000, 10000)`) and a much smaller one
   (`torch.randn(10, 10)`). Does relative error stay roughly constant, or
   change with scale? What does that imply for quantizing a model with a
   wide range of weight magnitudes across different layers?
3. **Chunk count vs stream overlap.** Re-run the streams benchmark with
   `N_CHUNKS = 2` and `N_CHUNKS = 32`. At what point does adding more
   streams stop helping? What resource do you think you're running out of?
